[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/tcia_utils_v4/TCIA_Dataset_and_Participant_Queries.ipynb)

# Finding datasets and building participant cohorts with TCIA

In this hands-on lab we'll start with a dataset, then work our way down to individual participants. Along the way we'll find images, clinical information, and segmentation labels. You only need a little Python and some familiarity with pandas tables.

We'll work through three questions:

1. **Which datasets have the data I need, and how do I download a whole dataset?**
2. **Can I find participants with an image modality, diagnosis, and primary disease site across datasets?**
3. **Which open-access CT images also have segmentation labels?**

Our main example is [CMB-GEC](https://www.cancerimagingarchive.net/collection/cmb-gec/), which includes radiology images, tissue-slide images, and links to clinical information. Later we'll select public CT scans and segmentation labels.

Use Python **3.11 or newer**. The installation cell sets up the packages used in this lab. This notebook is self-contained: you do not need other files from the repository.

Use a fresh output folder for the first run of this updated notebook; older download layouts are not converted automatically.

Running all cells queries real, public metadata, saves manifests and tables, and downloads the public image samples described below. Each public download uses `max_items=1` to select one whole series or file. Controlled downloads require your own approved access and remain off until you enable them. You do not need an LLM account to use this notebook.

## Let's get set up

Run the next cell to install the packages used in this lab. In Colab, run the notebook from the top. In Jupyter, use a fresh Python 3.11+ environment. Restart the Python session if prompted after installation.

`pandas` helps us work with tables. Download support is included, but querying the metadata does not download any images.

In [ ]:
import subprocess, sys
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "tcia_utils[tables,downloads]==4.0.0a17"]
)

In [ ]:
from pathlib import Path
from importlib.metadata import version
from datetime import datetime, timezone
from urllib.parse import urlsplit
import hashlib
import inspect
import json
import os

import httpx
import pandas as pd
from IPython.display import display, Markdown
import tcia_utils
from tcia_utils import TCIAClient, DownloadError

assert hasattr(tcia_utils, "format_table") and all(
    hasattr(TCIAClient, method)
    for method in ("download_manifest", "plan_participant_images", "download_images")
) and "primary_site" in inspect.signature(TCIAClient.search_participants).parameters and "accept_data_policy" in inspect.signature(TCIAClient.download_manifest).parameters, "Run the installation cell for tcia_utils 4.0.0a17, restart your Python/Colab runtime, then run from the top."
format_table = tcia_utils.format_table
pd.set_option("display.max_colwidth", 85)
OUTPUT = Path(os.environ.get("TCIA_EXAMPLE_OUTPUT_DIR", "tcia-workshop-files"))
OUTPUT.mkdir(parents=True, exist_ok=True)
BASE_URL = os.environ.get("TCIA_REST_BASE_URL", "https://tcia.duckdns.org")
tcia = TCIAClient(base_url=BASE_URL, timeout=120, pin_snapshot=True)
status = tcia.status()
SNAPSHOT = status.provenance.release_fingerprint
print("tcia_utils:", version("tcia_utils"))
print("Metadata observed:", datetime.now(timezone.utc).date())
print("Service ready:", status.data.readiness["status"])

TCIA's search service uses a prepared metadata snapshot so we can ask questions before downloading images. We'll keep original links with our results because the snapshot can lag behind a dataset's web page.

When we use `to_dataframe(all_pages=True)`, the package collects every search page. We also ask the client to keep the same snapshot throughout this session; it will tell us to restart if the metadata changes.

## 1. Which dataset should I download?

A **Collection** is a published dataset. An **Analysis Result** is another published contribution, often containing annotations or measurements made from existing images. Keep the dataset name and type together so we know exactly which contribution we're using.

There are also two independent choices to understand:

| Choice | What it means for us |
| --- | --- |
| Open access or controlled access | Open-access files can be retrieved under their license. Controlled-access files require an approved user account and agreement to the applicable data usage terms before download, even when their descriptions are public. |
| DICOM or non-DICOM | DICOM is a medical imaging format. Other useful formats include NIfTI image volumes, SVS tissue slides, spreadsheets, and archives. Non-DICOM does not mean non-image. |

A single collection can contain several combinations. **Open access** also does not mean unrestricted use: read each download's license, including any noncommercial condition.

Start in the [TCIA Dataset Explorer](https://www.cancerimagingarchive.net/dataset-explorer/). Try a cancer-location filter and a modality filter, then open a matching collection's page. For this lesson, choose **CMB-GEC**, open its **Data Access** tab, and note the separate download/manifest links and licenses. The Explorer helps us find the dataset; its page describes the individual download groups.

In Python, we can explore the same kinds of collection attributes. Let's try a cancer location and image modality. Change these values and rerun the cell to explore.

In [ ]:
CANCER_LOCATION = "Esophagus"
IMAGE_MODALITY = "CT"
datasets = tcia.search_datasets(
    dataset_type="collection",
    cancer_locations=[CANCER_LOCATION],
    modalities=[IMAGE_MODALITY],
    limit=100,
).to_dataframe(all_pages=True)
print("Matching collections:", len(datasets))
display(
    datasets.reindex(
        columns=[
            "short_title",
            "title",
            "data_categories",
            "download_data_types",
            "cancer_types",
            "cancer_locations",
            "resolved_access_level",
            "download_file_types",
            "external_resource_labels",
            "tcia_page",
        ]
    )
)

The `data_categories` column tells us what kinds of downloadable data a collection publishes, such as radiology or pathology images. `download_data_types` shows modalities and other data types, while `download_file_types` shows formats. External resources, such as linked clinical information, stay in their own column.

**What does “external” mean?** These are related data or tools hosted and maintained outside TCIA. The external materials have not been curated by TCIA staff, and TCIA does not control their content, updates, or availability. For example, a link may stop working if the organization hosting it has a site outage. Follow the resource's own instructions and contact its maintainers when you need help with it.

Multi-value fields are Python lists, even when there is only one value. An empty list means no value was reported.

These are **collection-level** labels. Finding a collection with CT and a cancer label does not tell us that every participant has both. We'll check participant-level information in section 2.

Now let's open CMB-GEC. The download table is especially useful: it tells us which files are **open access**, which are **controlled access** (requiring user accounts and data usage agreements), and which formats we'll receive.

We'll use `get_dataset()` for the dataset description and `plan_downloads()` to **review the download options**. A collection can have several download groups with different formats, access terms, and software requirements. `plan_downloads()` brings those details together and tells us which groups have a supported Python download route. It fetches metadata only: it does not download images, install software, or check whether we personally have access approval.

This review step helps us choose the groups we need before downloading. It is optional: `download_dataset()` does the same planning internally, and `download_manifest()` can use an existing manifest directly.

In [ ]:
DATASET = "CMB-GEC"
detail = tcia.get_dataset(DATASET)
collection = next((row for row in detail.data["datasets"] if row["dataset_type"] == "Collection"))
display(
    Markdown(
        f"### [{collection['short_title']}]({collection['tcia_page']})\n\n{collection['title']}"
    )
)
print("Dataset version:", collection["current_version_number"])
print("Public external clinical resource linked:", collection["has_external_clinical_resource"])
plan = tcia.plan_downloads(DATASET)
downloads = plan.to_dataframe()
display(
    format_table(
        downloads,
        columns=[
            "download_id",
            "download_title",
            "download_category",
            "download_requirements",
            "download_method",
            "python_supported",
            "subjects",
            "studies",
            "series",
            "images",
            "access_level",
            "data_types",
            "file_types",
            "license_label",
            "download_size",
            "download_size_unit",
            "download_url",
        ]
    )
)
downloads.to_csv(OUTPUT / "CMB-GEC_download_inventory.csv", index=False)

CMB-GEC is a good reminder that **a dataset is not always one download**. Its download categories distinguish radiology images, tissue-slide images, clinical tables, and other supporting files. Access and file format are separate attributes.

The download tables also show **patients/subjects, studies, series, and images** using the published `subjects`, `studies`, `series`, and `images` fields. These counts describe each **full download group**, not the small sample we'll try later. “Not reported” means the count is unavailable; it does not mean zero. Some counts do not apply to non-DICOM data. Groups can overlap, so adding their counts does not establish a total number of unique patients.

The download plan includes a `download_requirements` column describing the published download instructions. `python_supported` tells us which groups this package can transfer using Python and pip-installed dependencies. These are separate from access approval: controlled files still require your approved key. Groups needing another application remain in the inventory with their official links, but this notebook does not install or run those applications. A manifest is a shopping list; it does not contain the images or grant access to controlled files.

`download_method` describes the kind of download link. It helps us choose the right next step:

| `download_method` | What the link gives us |
| --- | --- |
| `dicom_manifest` | A list of DICOM series to download. |
| `tabular_manifest` | A table listing files to download. |
| `direct_file` | The file itself, which may be an archive. |
| `aspera_package` | A package requiring a separate transfer application. |
| `unsupported` | A link without a supported Python download route. |

The next cell keeps `dicom_manifest` and `tabular_manifest` in `manifest_groups` because `save_manifests()` saves these small lists. We leave direct-file and package links in the inventory; saving those links as if they were manifests would not give us a list of images.

In [ ]:
download_plan = downloads
python_groups = download_plan[download_plan["python_supported"]]
external_groups = download_plan[~download_plan["python_supported"]]
manifest_groups = python_groups[
    python_groups["download_method"].isin(["dicom_manifest", "tabular_manifest"])
]
display(
    format_table(
        download_plan,
        columns=[
            "download_id",
            "download_title",
            "download_category",
            "download_requirements",
            "download_method",
            "python_supported",
            "subjects",
            "studies",
            "series",
            "images",
            "access_level",
            "file_types",
        ]
    )
)
print("Selected download groups:", len(download_plan))
print("Groups supported by Python:", len(python_groups))
print("Groups requiring an external download route:", len(external_groups))

### Save the official manifests

The next cell uses `tcia.save_manifests()` to save the small official manifest files, with their original contents, source links, and checksums. It does not download images. Package links stay in the inventory; they are not mistaken for manifests.

**Controlled access:** the classification in the table comes from the download's license metadata. Public descriptions and manifests are not authorization. Read the [current TCIA access policy](https://www.cancerimagingarchive.net/nih-controlled-data-access-policy/), request the required dbGaP authorization, and create your own JSON API key using the linked instructions. For CMB-GEC's controlled images the policy identifies **phs002192**. You can still save and inspect the public manifest without that key.

In [ ]:
manifest_report = tcia.save_manifests(manifest_groups, OUTPUT)
saved_manifests = manifest_report.records
display(manifest_report.to_dataframe())

### Download directly from Python

Let's start with the **public DICOM imaging group**. We already saved its manifest, so we can pass that file straight to `tcia.download_manifest()`. The package reads the identifiers and handles the transfer; we do not need to parse the file ourselves. It prints “Preparing download selection…” immediately, then shows periodic transfer-progress lines in the cell output. These normally arrive about every five seconds, with byte counts or percentages when available, and a final result. A short transfer may finish before a periodic update. The setup cell installs the transfer package; download calls do not install software.

This function accepts a **local path or an HTTPS URL**, for either a `.tcia` manifest or a supported CSV/TSV manifest. It reads the manifest's route column and chooses the appropriate download method. If part of the requested selection is unavailable, it tells us rather than silently dropping those files.

Before running a download cell, read the [TCIA data usage policy](https://www.cancerimagingarchive.net/data-usage-policies-and-restrictions/). The calls below use `accept_data_policy=True` to record your acceptance. Run them only if you agree to that policy.

Before a large download, let's **try one item**. Passing `max_items=1` downloads the first whole DICOM series or file in the manifest. This is a real transfer, so it can help us check the setup and inspect the data. A whole series or file can still be large: this option limits the number of items, not the number of bytes.

Run the next cell to download one DICOM series into a sample folder. The report shows how many items were selected and how many were available. These are the first items in the supplied order, so they may not represent every kind of data in the collection.

After checking the sample, **keep the same destination and remove `max_items=1`** to request the full manifest. The transfer package can reuse the series already present; it checks and reuses those files automatically. Choose a separate folder only if you want a separate copy.

We choose `OUTPUT / "samples"` as the download root. The folder name is our choice: we can keep using it for the full download. Public DICOM images are organized underneath it by collection, participant, study, and series. We do not need to add those details to the destination ourselves. Small transfer lists and transfer metadata may appear alongside the collection folders.

`Synced` in the public DICOM summary identifies existing-series sync jobs; those jobs can still fetch missing files. To retry a download, rerun the same call with the same destination. An incomplete file may need to be transferred again. A successful sample does not establish that every remaining item is available or that an image and segmentation match.

In [ ]:
public_manifest = next(
    item for item in saved_manifests if item["route_column"] == "SeriesInstanceUID"
)

report = tcia.download_manifest(
    OUTPUT / public_manifest["manifest"], OUTPUT / "samples", max_items=1, accept_data_policy=True
)
display(report.to_dataframe())

### Groups requiring another application

The tissue-slide group in CMB-GEC needs a separate transfer application, so it stays outside this notebook's Python downloads. Its counts, published requirements, and official link remain in the inventory above. Follow that link or use our [separate package-download notebook](https://github.com/kirbyju/TCIA_Notebooks/blob/main/TCIA_Aspera_CLI_Downloads.ipynb) when you need those original files.

Other collections may offer non-DICOM files through Python-supported manifests or HTTPS links. Use `python_supported` to check each group. Downloading only the Python-supported groups does **not** mean we have downloaded the entire collection.

### Download the controlled group after approval

For controlled images, the Python function also needs the **path** to your own JSON API key. It uses the official file identifiers and the authorized service. The transfer package uses the supplied key for this download invocation only; it does not save a credential copy or reuse a desktop application's saved key. See its [credential guidance](https://github.com/TCIA/data-retriever/blob/2dcad4bd6fe0bd62b42e67834d0ad42240694f42/docs/cli-progress-and-credentials.md). Don't paste key contents into a notebook, repository, or chat.

Leave the next cell unchanged to continue without a controlled download. An approved user can enter a key-file path and set `DOWNLOAD_CONTROLLED_CMB = True` to try the first file in the manifest. It still needs the same authorization as a larger download. Authorization for one study may not cover another.

These downloads may be ZIP archives with their original filenames. A standalone manifest call uses the generic folder label `manifest`, so its layout can differ from the public DICOM folders above. This file-download route leaves ZIPs as archives; it does not automatically unpack them into the same DICOM hierarchy.

After inspecting the sample, keep the same destination and remove `max_items=1` for the full manifest. Existing-file checks use the supplied size/checksum when available. Files that pass the existing-file checks are counted as `Skipped` and still count toward completed progress. `Downloaded` counts new file transfers. `Synced` is used for the public DICOM synchronization jobs described above; controlled-file reuse does not need that category.

In [ ]:
CREDENTIAL_FILE = ""  # Path to your own approved JSON key file; never the key value.
DOWNLOAD_CONTROLLED_CMB = False
controlled_manifest = next(item for item in saved_manifests if item["route_column"] == "drs_uri")

if DOWNLOAD_CONTROLLED_CMB:
    report = tcia.download_manifest(
        controlled_manifest["url"],  # A URL works just like a local manifest path.
        OUTPUT / "samples",
        credential_file=CREDENTIAL_FILE,
        accept_data_policy=True,
        max_items=1,
    )
    display(report.to_dataframe())
else:
    print("Controlled CMB-GEC download is off. Public manifests are ready for an approved user.")

Before calling a dataset download complete, compare your downloaded files and the SDK completion report with **every row** in the inventory. Resolve failed or skipped files, follow the separate download route for the tissue-slide group, and follow the collection's external-resource links if your project needs those data too. A manifest on disk or a successful metadata query is not evidence that images were transferred.

Let's save the dataset citation as well. Keep the citation and license information with your project, and follow the collection page's acknowledgement instructions.

In [ ]:
citation = tcia.get_citation(collection["doi"], style="apa")
print(citation.data["record"]["formatted_citation"])
(OUTPUT / "CMB-GEC_citation.json").write_text(json.dumps(citation.data, indent=2))

## 2. Which participants match my modality, diagnosis, and disease site?

Suppose we want **CT** images from participants whose available **diagnosis includes “Carcinoma”** and whose **primary disease site includes “Lung.”** These three conditions must all match. We'll search CMB-LCA and NSCLC-Radiomics to find matches across datasets. We also include CMB-GEC so we can see how the disease-site filter narrows the selection.

Choose the collection list, modality, diagnosis, and primary disease site together in the next cell. The search requires **all three conditions** before returning any participants. No images are downloaded by this query. For example, try `PRIMARY_SITE_TEXT = "Esophagus"` to look for gastroesophageal participants instead. Disease site refers to the clinical `primary_site` field, not the body part covered by the CT scan. A scan may cover several regions or show metastases.

Leave `INCLUDE_INFERRED_CLINICAL = True` to allow supported clinical labels inferred from the dataset. Set it to `False` to require both matching clinical values to be recorded for the participant, with patient-level source evidence. Unknown inference status is excluded in that mode. The results retain separate diagnosis and site inference flags so we can inspect their evidence.

The text filters use case-insensitive substring matching. They do not expand medical synonyms: “Lung” can match “upper lobe of right lung,” but a different term such as “pulmonary” is not automatically equivalent. Start with these selected collections and inspect the reported labels before searching more broadly.

In [ ]:
COHORT_COLLECTIONS = ["CMB-GEC", "CMB-LCA", "NSCLC-Radiomics"]
COHORT_MODALITY = "CT"
DIAGNOSIS_TEXT = "Carcinoma"
PRIMARY_SITE_TEXT = "Lung"
INCLUDE_INFERRED_CLINICAL = True

participants = tcia.search_participants(
    short_titles=COHORT_COLLECTIONS,
    dataset_type="collection",
    modalities=[COHORT_MODALITY],
    diagnosis=DIAGNOSIS_TEXT,
    primary_site=PRIMARY_SITE_TEXT,
    include_inferred=INCLUDE_INFERRED_CLINICAL,
    limit=500,
).to_dataframe(all_pages=True)
if participants.empty:
    participants = pd.DataFrame(
        columns=["short_title", "display_participant_id", "participant_key", "modalities"]
    )
    print("No participants matched. Try another collection, modality, diagnosis, or disease site.")
print("Participants matching modality, diagnosis AND primary site:", len(participants))
display(
    participants.reindex(
        columns=[
            "short_title",
            "display_participant_id",
            "modalities",
            "file_formats",
            "has_open_data",
            "has_controlled_data",
        ]
    ).head(10)
)

Clinical information can come from a collection's own files or public clinical enrichment from other resources. The API keeps the source and whether a value was **recorded for the participant** or **inferred from the dataset**. For example, a collection-level cancer label is useful for discovery, but it is not a confirmed diagnosis for each person.

`tcia.clinical_for_participants()` handles the batching and checks the returned identities. We'll join exact participant IDs **within each collection**. We won't guess an ID conversion or assume that the same ID in two collections means the same person. Unmatched IDs stay visible.

In [ ]:
clinical_result = tcia.clinical_for_participants(participants)
clinical = clinical_result.to_dataframe(
    "subjects",
    columns=[
        "short_title",
        "subject_id",
        "primary_diagnosis",
        "primary_diagnosis_is_inferred",
        "primary_site",
        "primary_site_is_inferred",
        "resolved_values",
        "resolved_sources",
        "conflicts",
    ],
)
with_clinical = participants.merge(
    clinical,
    how="left",
    left_on=["short_title", "display_participant_id"],
    right_on=["short_title", "subject_id"],
    validate="many_to_one",
    indicator="clinical_link",
)
print("Clinical linkage (no fuzzy ID matching):")
display(with_clinical["clinical_link"].value_counts())
print("Available diagnosis labels:")
display(with_clinical["primary_diagnosis"].value_counts(dropna=False).to_frame("participants"))
print("Available primary disease-site labels:")
display(with_clinical["primary_site"].value_counts(dropna=False).to_frame("participants"))

Our initial search already applied the modality, diagnosis, and disease-site criteria. We've now added the full clinical records for those matches; we do not need to filter the table again.

The displayed and exported rows retain separate `primary_diagnosis_is_inferred` and `primary_site_is_inferred` columns. `True` means inferred, `False` means not inferred, and a blank means unknown. Source evidence tells us whether a value is reported for the participant. To change the criteria or inference setting, rerun this section from its first code cell.

We are counting **dataset-scoped participant records**, not unique people deduplicated across collections. Missing clinical values do not establish the absence of disease. This cohort covers only the selected collections and the information currently indexed.

In [ ]:
cohort = with_clinical.copy()
print("Final cohort matching modality, diagnosis AND primary site:", len(cohort))
display(
    cohort.reindex(
        columns=[
            "short_title",
            "display_participant_id",
            "modalities",
            "primary_diagnosis",
            "primary_diagnosis_is_inferred",
            "primary_site",
            "primary_site_is_inferred",
            "has_open_data",
            "has_controlled_data",
        ]
    )
)
cohort.to_json(OUTPUT / "modality_diagnosis_site_cohort.json", orient="records", indent=2)
cohort.drop(columns=["resolved_values", "resolved_sources", "conflicts"]).to_csv(
    OUTPUT / "modality_diagnosis_site_cohort.csv", index=False
)

### What else do we know about these participants?

The short table above is only a starting point. Let's put **all resolved clinical concepts** for the matched participants into a second table. That can include demographics, outcomes, treatment-related values, or molecular findings when those were indexed. We keep the source evidence beside each value rather than assuming all collections report the same things.

These clinical summaries are a **derived curation layer**. They do not replace the [original dataset descriptions and clinical files](https://www.cancerimagingarchive.net/collections/). CMB-GEC links additional clinical resources on its page; the API includes the public facts it has indexed, not every field available at those links or any restricted clinical file. Blank means “not available here,” not “does not exist.”

In [ ]:
clinical_values = clinical_result.to_dataframe(
    "values",
    columns=[
        "short_title",
        "participant_id",
        "concept",
        "value",
        "is_inferred",
        "evidence_scope",
        "source_evidence",
    ],
)
clinical_summary = clinical_values.merge(
    cohort[["short_title", "display_participant_id"]],
    left_on=["short_title", "participant_id"],
    right_on=["short_title", "display_participant_id"],
    how="inner",
    validate="many_to_one",
).drop(columns="display_participant_id")
display(clinical_summary.drop(columns="source_evidence").head(25))
clinical_summary.to_json(OUTPUT / "cohort_clinical_evidence.json", orient="records", indent=2)
print(
    "The displayed rows are a preview; the JSON export contains every resolved value collected above."
)

For a closer look, choose one matching participant. The first cell lists their known data types, access levels, formats, and geometry summaries. The next keeps the original clinical facts, source links, and disagreements for that participant. We use CMB-GEC first when it is present in the cohort.

In [ ]:
ordered = cohort.assign(example_first=cohort["short_title"].ne("CMB-GEC")).sort_values(
    ["example_first", "short_title", "display_participant_id"]
)
example = ordered.iloc[0].to_dict() if len(ordered) else None
participant_assets = pd.DataFrame()
if example:
    print(example["short_title"], example["display_participant_id"])
    participant = tcia.get_participant(example["participant_key"])
    participant_assets = tcia.list_participant_assets(
        example["participant_key"], limit=500
    ).to_dataframe(all_pages=True)
    display(
        participant_assets.reindex(
            columns=[
                "data_category",
                "data_type",
                "modality",
                "access_level",
                "file_format",
                "file_count",
                "geometry_status",
            ]
        )
    )
    (OUTPUT / "example_participant.json").write_text(json.dumps(participant.data, indent=2))
    participant_assets.to_json(
        OUTPUT / "example_participant_assets.json", orient="records", indent=2
    )
else:
    print("No matches. Try diagnosis and primary-site labels from the tables above.")

In [ ]:
example_facts = []
if example:
    facts = tcia.list_clinical_facts(
        example["short_title"], subject_id=example["subject_id"], limit=500
    )
    conflicts = tcia.list_clinical_conflicts(
        example["short_title"], subject_id=example["subject_id"], limit=500
    )
    example_facts = facts.data["facts"]
    display(
        pd.DataFrame(example_facts).reindex(
            columns=[
                "concept",
                "value_text",
                "value_resolved",
                "is_inferred",
                "evidence_scope",
                "source_url",
                "source_date",
                "qc_status",
            ]
        )
    )
    print("Clinical conflict rows:", conflicts.data["count"])
    if conflicts.data["count"]:
        display(pd.DataFrame(conflicts.data["conflicts"]))
    print(
        "Facts returned:",
        facts.data["count"],
        "(bounded at 500; this is not an all-facts pagination route)",
    )
    (OUTPUT / "example_clinical_facts.json").write_text(json.dumps(facts.data, indent=2))
    (OUTPUT / "example_clinical_conflicts.json").write_text(json.dumps(conflicts.data, indent=2))

### From this clinical cohort to radiology images

Now that we have our participant list, let's use it to choose some images. We'll keep the **CMB-LCA participants in `cohort`**, so the clinical criteria we just used still apply.

For radiology, ask for **CT in DICOM format**. `plan_participant_images()` resolves the matching public series and reports coverage for each selected participant, including those with no matches. The same function will select non-DICOM pathology files in the next example. Metadata sources and versions stay with the plan; we do not need to choose a separate query service.

The next cell inspects the plan and downloads a sample. `max_items=1` selects **one whole CT series**, including its image files. Omit that limit after reviewing the plan to download all selected public CT series. A single series can still be large. If a participant identifier or route cannot be resolved, the plan explains why and the download stops for review.

In [ ]:
clinical_image_participants = cohort[
    cohort["short_title"].eq("CMB-LCA") & cohort["dataset_type"].str.casefold().eq("collection")
].copy()
ct_images = tcia.plan_participant_images(
    clinical_image_participants, modality="CT", file_format="DICOM"
)
display(ct_images.coverage_to_dataframe())
display(ct_images.to_dataframe().reindex(
    columns=["short_title", "participant_id", "study_instance_uid", "series_instance_uid",
             "modality", "file_format", "instance_count", "size_bytes", "provider"]
))
if ct_images.unresolved:
    display(pd.DataFrame(ct_images.unresolved))
    print("Image selection is unresolved; review these issues before downloading.")
elif ct_images.rows:
    report = tcia.download_images(ct_images, OUTPUT / "clinical-ct-sample", max_items=1, accept_data_policy=True)
    display(report.to_dataframe())
else:
    print("No matching public CT series were found.")

### Use the same clinical cohort for whole-slide pathology images

For a pathology project, keep the **same CMB-LCA participant list** and ask for **Whole Slide Image in non-DICOM format**. This gives us individual slide files rather than a full-collection package. The coverage table tells us which clinically matched participants have downloadable slides; a participant with CT is not guaranteed to have a slide here.

These are **public slide copies**, which may have been reformatted for browser viewing. Their equivalence to the submitter's original files is not established. If your analysis requires the submitted originals, follow the original-file download on the [CMB-LCA collection page](https://www.cancerimagingarchive.net/collection/cmb-lca/). That separate application workflow is outside this notebook.

The next cell plans all matching public slides and uses `max_items=1` to download **one whole slide file**. Omit that limit to download all matching slide files in the plan. Whole-slide files can be large even when we're downloading only one. Any matched file without a supported route remains unresolved and stops the transfer; we do not silently skip it.

In [ ]:
clinical_image_participants = cohort[
    cohort["short_title"].eq("CMB-LCA") & cohort["dataset_type"].str.casefold().eq("collection")
].copy()
slide_images = tcia.plan_participant_images(
    clinical_image_participants, modality="Whole Slide Image", file_format="non-DICOM"
)
display(slide_images.coverage_to_dataframe())
display(
    slide_images.to_dataframe().reindex(
        columns=["short_title", "participant_id", "modality", "file_format", "file_name",
                 "size_bytes", "download_representation", "python_supported"]
    )
)
if slide_images.unresolved:
    display(pd.DataFrame(slide_images.unresolved))
    print("Image selection is unresolved; review these issues before downloading.")
elif slide_images.rows:
    report = tcia.download_images(slide_images, OUTPUT / "clinical-slide-sample", max_items=1, accept_data_policy=True)
    display(report.to_dataframe())
else:
    print("No matching public slides were found.")

## 3. Which open-access CT images have segmentation labels?

Let's finish with [NSCLC-Radiomics](https://www.cancerimagingarchive.net/collection/nsclc-radiomics/). We'll look for public CT scans and DICOM **SEG** segmentation images for the same participant and study. These downloads do not require a controlled-access account or key.

A segmentation labels regions in an image, such as a tumor. Finding CT and SEG in the same study gives us a useful starting point. It does not prove which CT series the SEG references. Check those DICOM references and the segment names before using the pair in an analysis.

In [ ]:
SEG_COLLECTION = "NSCLC-Radiomics"
seg_candidates = tcia.search_participants(
    short_titles=[SEG_COLLECTION], dataset_type="collection",
    modalities=["SEG"], file_formats=["DICOM"], access_levels=["open"], limit=5,
).to_dataframe()
print("Candidate participants on this page:", len(seg_candidates))
# Start with two participants so we can inspect the results. This is a preview,
# not a count of everyone in the collection.
seg_participants = seg_candidates.head(2).copy()
public_ct_images = tcia.plan_participant_images(
    seg_participants, modality="CT", file_format="DICOM"
)
public_seg_images = tcia.plan_participant_images(
    seg_participants, modality="SEG", file_format="DICOM"
)
if public_ct_images.unresolved or public_seg_images.unresolved:
    raise RuntimeError("Some participant image routes are unresolved; inspect the plans before downloading.")
ct_seg_files = pd.concat(
    [public_ct_images.to_dataframe(), public_seg_images.to_dataframe()], ignore_index=True
)
if ct_seg_files.empty:
    ct_seg_files = pd.DataFrame(columns=[
        "short_title", "participant_id", "study_instance_uid", "series_instance_uid", "modality",
    ])
usable = ct_seg_files.dropna(subset=["participant_id", "study_instance_uid"]).copy()
modalities_by_study = usable.groupby(
    ["short_title", "participant_id", "study_instance_uid"]
)["modality"].agg(set)
matching_studies = modalities_by_study[
    modalities_by_study.apply(lambda values: {"CT", "SEG"}.issubset(values))
]
ct_seg_cohort = matching_studies.reset_index(name="available_modalities")
display(ct_seg_cohort)
ct_seg_cohort.to_json(OUTPUT / "public_ct_seg_cohort.json", orient="records", indent=2)

Choose one participant-study pair, then keep one CT series and one SEG series from it. The manifest below is a small CSV listing those exact series identifiers. It is separate from the full-dataset manifest we used earlier.

For collections with multiple CT or SEG series in one study, inspect the descriptions and references rather than assuming the first pair is the right one for your question.

In [ ]:
ct_seg_selection = pd.DataFrame()
ct_seg_manifest = OUTPUT / "public_ct_seg_sample.csv"
if len(ct_seg_cohort):
    chosen = ct_seg_cohort.sort_values(["participant_id", "study_instance_uid"]).iloc[0]
    ct_seg_selection = usable[
        usable["short_title"].eq(chosen["short_title"])
        & usable["participant_id"].eq(chosen["participant_id"])
        & usable["study_instance_uid"].eq(chosen["study_instance_uid"])
    ].sort_values(["modality", "series_instance_uid"]).groupby("modality").head(1)
    ct_seg_selection[["series_instance_uid"]].rename(
        columns={"series_instance_uid": "SeriesInstanceUID"}
    ).to_csv(ct_seg_manifest, index=False)
    display(ct_seg_selection.reindex(columns=[
        "short_title", "participant_id", "study_instance_uid", "series_instance_uid",
        "modality", "instance_count", "size_bytes", "license_label",
    ]))
    print("Selected CT/SEG series:", len(ct_seg_selection))
else:
    print("No same-study CT/SEG pair matched this preview. Try another participant page.")

Now download this small selection. Here `max_items=2` means **two series**, so it keeps both the scan and the segmentation. It does not limit the number of DICOM images or the number of bytes. As before, rerunning the same call with the same destination lets Data Retriever check and reuse files already present.

In [ ]:
if ct_seg_selection.empty:
    raise ValueError("No eligible public CT/SEG sample was selected.")
report = tcia.download_manifest(
    ct_seg_manifest, OUTPUT / "public-ct-seg",
    accept_data_policy=True, max_items=2,
)
display(report.to_dataframe())

### What about NIfTI files delivered through Aspera?

The [Aspera download notebook](https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/TCIA_Aspera_CLI_Downloads.ipynb) shows how to find a brain MR cohort with NIfTI segmentations, export the participants' exact file paths with `tcia_utils`, and download a small sample. It also accepts a saved participant cohort from another notebook, such as our `modality_diagnosis_site_cohort.json` file, when those participants have indexed Aspera files.

That notebook includes the separate Aspera installation and an optional image/mask geometry check.

## Keep a record of what you used

We've gone from collection discovery to whole-dataset download planning, then built participant cohorts using modality, diagnosis, and segmentation availability. The saved tables include all search pages, while some displayed tables are deliberately short previews. The candidate and clinical-fact previews have explicit limits; they are not exhaustive archive-wide queries.

Before using a cohort in a study, review its source links, missing clinical values, inferred diagnoses, exact image/label relationships, and applicable licenses. Keep the dataset citations and split related scans at the person level when the dataset documents those relationships.

The next cell saves our query settings, metadata snapshot identity, original manifest URLs/checksums, and counts beside the exported tables. **It records metadata work, not proof that any image transfer completed.**

In [ ]:
run_record = {
    "observed_at_utc": datetime.now(timezone.utc).isoformat(),
    "tcia_utils_version": version("tcia_utils"),
    "metadata_snapshot": SNAPSHOT,
    "query": {
        "collections": COHORT_COLLECTIONS,
        "modality": COHORT_MODALITY,
        "diagnosis_text": DIAGNOSIS_TEXT,
        "primary_site_text": PRIMARY_SITE_TEXT,
        "include_inferred": INCLUDE_INFERRED_CLINICAL,
    },
    "dataset_download_groups": len(download_plan),
    "clinical_ct_series": len(ct_images.rows),
    "clinical_slide_files": len(slide_images.rows),
    "clinical_ct_index_version": ct_images.providers.get("IDC", {}).get("idc_version"),
    "image_planning_providers": ct_images.providers,
    "saved_manifests": saved_manifests,
    "external_download_ids": external_groups["download_id"].tolist(),
    "search_match_records": len(participants),
    "matching_records": len(cohort),
    "public_seg_collection": SEG_COLLECTION,
    "public_ct_seg_studies": len(ct_seg_cohort),
    "public_ct_seg_selected_series": len(ct_seg_selection),
    "scope": "Dataset-scoped records; no cross-dataset person deduplication.",
    "clinical_caveat": "Derived clinical curation; original dataset descriptions and source clinical files remain authoritative.",
    "payload_completion_verified": False,
    "source_pages": [
        collection["tcia_page"],
        "https://www.cancerimagingarchive.net/collection/nsclc-radiomics/",
        "https://www.cancerimagingarchive.net/nih-controlled-data-access-policy/",
    ],
}
(OUTPUT / "workshop_run.json").write_text(json.dumps(run_record, indent=2))
tcia.close()
print("Saved tables, manifests, and workshop_run.json in the notebook's output folder.")

### Try your own question

Change `COHORT_COLLECTIONS`, `COHORT_MODALITY`, `DIAGNOSIS_TEXT`, and `PRIMARY_SITE_TEXT`, then rerun section 2 onward. Start with a small set of collections so you can inspect their clinical labels and coverage before widening the search.

For another challenge, use the Aspera notebook to explore NIfTI image and mask geometry for a selected participant cohort.

For advanced public DICOM metadata queries and viewers beyond this tutorial, [NCI Imaging Data Commons (IDC)](https://learn.canceridc.dev/) provides its own tools. The shared image plan retains IDC attribution when its metadata is used.

Useful links: [TCIA Dataset Explorer](https://www.cancerimagingarchive.net/dataset-explorer/), [CMB-GEC](https://www.cancerimagingarchive.net/collection/cmb-gec/), [NSCLC-Radiomics](https://www.cancerimagingarchive.net/collection/nsclc-radiomics/), [controlled-access instructions](https://www.cancerimagingarchive.net/nih-controlled-data-access-policy/).